# Импорты

In [161]:
from IPython.display import clear_output     # очистка вывода ячейки (попытка создания непрерывной "анимации")
from datetime import datetime as dt          # получение времени и даты для вставки в название лога
import random as rnd                         # создание "случайностей"
import time                                  # для блокировки потока (попытка фиксировать FPS)

# Вспомогательные функции

In [162]:
def choice_prob(more_prob, less_prob, prob=0.8):
    if more_prob and less_prob:
        weights = [prob / len(more_prob)] * len(more_prob) + [(1 - prob) / len(less_prob)] * len(less_prob)
    elif less_prob:
        weights = [1 / len(less_prob)] * len(less_prob)
    else:
        weights = [1 / len(more_prob)] * len(more_prob)

    return rnd.choices(more_prob + less_prob, weights=weights, k=1)[0]

def die_pred(current, max_stat):
    if current >= 2 * max_stat:
        return True
    death_chance = (current / max_stat) ** 2
    return rnd.random() < death_chance

def sex_choice():
    while True:
        yield "XX"
        yield "XY"

sex_choiser = sex_choice()
sex_gen = lambda: next(sex_choiser)

crusade = lambda x, y: [(x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)]

circle = lambda x, y, r: [(i, j) for i in range(x - r, x + r + 1) for j in range(y - r, y + r + 1) if (i, j) != (x, y)]

find_parent_name = lambda obj: obj.__class__.__bases__[0].__name__

find_class_name = lambda obj: obj.__class__.__name__

# Базовые классы

In [163]:
class CelestialObj:
    def __init__(self):
        self.ticks = 0
        self.time = "Day 0 | 0:00 (Night)"
    
    def tick(self):
        self.ticks += 1
        self.time = f"Day {self._get_day()} | {self._get_time()}:00 ({self._get_pos()})"

    def _get_day(self):
        return self.ticks // 24

    def _get_time(self):
        return self.ticks % 24
    
    def _get_pos(self):
        return ["Night", "Morning", "Day", "Afternoon"][((self.ticks % 24 + 1) // 6) % 4]

class World():
    def __init__(self, size, cel_obj=None):
        if cel_obj is None:
            cel_obj = CelestialObj()
            
        self.width = size[0]
        self.height = size[1]
        self.cel_obj = cel_obj
        self.entities = {}
        self.plain = [["|   " for _ in range(self.width + 1)] for _ in range(self.height)]
        self.free_places = [(x, y) for x in range(self.width) for y in range(self.height)]
        
    def add_entity(self, Entity):
        self.entities[Entity.coord] = Entity
        self.free_places.remove(Entity.coord)
        x, y = Entity.coord
        self.plain[y][x] = Entity.mark
    
    def del_entity(self, Entity):
        self.entities.pop(Entity.coord)
        self.free_places.append(Entity.coord)
        x, y = Entity.coord
        self.plain[y][x] = "|   "

    def repl_entity(self, New_entity):
        self.entities[New_entity.coord] = New_entity
        x, y = New_entity.coord
        self.plain[y][x] = New_entity.mark
    
    def move_entity(self, Entity, new_place):
        self.entities.pop(Entity.coord)
        self.free_places.append(Entity.coord)
        x, y = Entity.coord
        self.plain[y][x] = "|   "
        self.entities[new_place] = Entity

        if new_place in self.free_places:
            self.free_places.remove(new_place)
            
        x, y = new_place
        self.plain[y][x] = Entity.mark

    def __contains__(self, obj):
        if isinstance(obj, tuple):
            return 0 <= obj[0] < self.width and 0 <= obj[1] < self.height
        return obj in self.entities.values()

class GameLoop:
    def __init__(self, entities_list=[], world_size=(10, 10)):
        self.world = World(world_size)

        for Entity, num in entities_list:
            for _ in range(num):
                place = rnd.choice(self.world.free_places)
                self.world.add_entity(Entity(self.world, place))

        print(self._render_frame())

    def _tick(self):
        self.world.cel_obj.tick()

        entities = list(self.world.entities.values())
        rnd.shuffle(entities)

        for Entity in entities:
            Entity.live()

    def _render_frame(self):
        time_state = self.world.cel_obj.time
        borders = "...." * self.world.width + "."
        plain = "\n".join(["".join(row) for row in self.world.plain])
        frame = "\n".join([time_state, borders, plain, borders])
        return frame
    
    def _log(self, frame, file):
        with open(f"{file}.txt", "a", encoding="utf-8") as file:
            file.write(frame + "\n")

    def loop(self, ticks=24, delay=0.2, logging=False, log_filename=None, clickable=False):
        if log_filename is None:
            log_filename = dt.now().strftime(r"%H-%M-%S_%d-%m-%Y")

        for _ in range(ticks):
            clear_output(wait=True)

            frame = self._render_frame()
            if logging: self._log(frame, log_filename)
            print(frame)
            self._tick()

            time.sleep(delay)
            if clickable:
                input()

# Сущности

In [164]:
class Entity:
    def __init__(self, world, coord):
        self.world = world
        self.coord = coord

In [165]:
PLANT_REGISTRY = {}
ANIMAL_REGISTRY = {}

In [166]:
class EcosystemMeta(type):
    def __new__(mcs, name, bases, namespace):
        bases_names = [base.__name__ for base in bases]

        cls = super().__new__(mcs, name, bases, namespace)

        if "Plant" in bases_names and name != "Plant":
            PLANT_REGISTRY[name] = cls
        elif "Animal" in bases_names and name != "Animal":
            ANIMAL_REGISTRY[name] = cls

        cls.env_state = 'time'

        cls.mark = ""
        cls.active_time = []
        cls.age = 0

        def _do_nothing(self):
            pass

        def _die(self):
            self.world.del_entity(self)

        def _lifecycle(self):
            pass

        def live(self):
            self._adapt()
            self._lifecycle()

        cls._do_nothing = _do_nothing
        cls._die = _die
        cls._lifecycle = _lifecycle
        cls.live = live

        if "Plant" in bases_names:
            def _adapt(self):
                die = die_pred(self.age, self.max_age)
                self.age += 1
                
                if not (self.world.cel_obj._get_pos() in self.active_time):
                    self._lifecycle = self._die if die else self._do_nothing
                elif die:
                    self._lifecycle = self._die
                else:
                    self._lifecycle = self._grow
                
            def _grow(self):
                if rnd.randint(0, 100) > self.grow_prob:
                    return

                coords = crusade(*self.coord)

                places = []
                prob_places = []

                for coord in coords:
                    if not coord in self.world:
                        continue

                    if coord in self.world.free_places:
                        places.append(coord)
                        continue
                        
                    entity = self.world.entities[coord]
                    same_parent = find_parent_name(entity) == find_parent_name(self)
                    diff_class = find_class_name(entity) != find_class_name(self)
                    
                    if same_parent and diff_class:
                        prob_places.append(coord)

                if places and prob_places:
                    place = choice_prob(places, prob_places, self.repl_prob / 100)
                elif places:
                    place = rnd.choice(places)
                elif prob_places and rnd.randint(0, 100) < self.repl_prob:
                    place = rnd.choice(prob_places)
                else:
                    return
                
                new_plant = self.__class__(self.world, place)

                if place in self.world.free_places:
                    self.world.add_entity(new_plant)
                else:
                    self.world.repl_entity(new_plant)

            cls._adapt = _adapt
            cls._grow = _grow
            
            cls.grow_prob = 60
            cls.repl_prob = 3
            cls.max_age = 48

        elif "Animal" in bases_names:
            def _adapt(self):
                if self.world.cel_obj.ticks == 1:
                    self._lifecycle = self._repr
                    return
                
                self.swarm = self._find_swarm()
                self.aggr = len(self.swarm)
                self.age += 1

                age = die_pred(self.age, self.max_age)
                hunger = die_pred(self.hunger, self.max_hunger)
                die = max(age, hunger)

                if not (self.world.cel_obj._get_pos() in self.active_time):
                    self._lifecycle = self._do_nothing
                elif die:
                    self._lifecycle = self._die
                else:
                    if self.hunger < self.max_hunger * 2 // 3 and self._find_pair() and self.sex == "XX" and self.age > 12:
                        action = self._repr
                        self.hunger = self.max_hunger // 3
                    elif self.hunger > (self.max_hunger // 10) or self.aggr >= self.max_swarm:
                        action = self._eat
                    else:
                        action = self._move_to_swarm

                    def lifecycle():
                        action()
                        self.hunger += 1

                    self._lifecycle = lifecycle

            def _eat(self):
                target = self._hunt()

                if target is None:
                    self._move()
                else:
                    self.world.del_entity(self.world.entities[target])
                    self._move(target)
                    self.hunger = 0

            def _hunt(self):
                if self.aggr > self.max_swarm:
                    self.swarm.remove(self.coord)
                    return rnd.choice(self.swarm)

                observed = circle(*self.coord, r=2)
                rnd.shuffle(observed)

                for coord in observed:
                    if coord in self.world.entities.keys():
                        target_class = find_class_name(self.world.entities[coord])
                        if target_class in self.diet:
                            return coord

                return None
            
            def _move(self, target=None):
                if target is None:
                    coords = crusade(*self.coord)
                    places = [self.coord]

                    for coord in coords:
                        if not coord in self.world:
                            continue
                        elif coord in self.world.free_places or find_parent_name(self.world.entities[coord]) != find_parent_name(self):
                            places.append(coord)

                    target = rnd.choice(places)

                self.world.move_entity(self, target)
                self.coord = target

            def _move_to_swarm(self):
                target = rnd.choice(self.swarm)
                places = circle(*target, r=2)
                rnd.shuffle(places)

                for place in places:
                    if place in self.world.free_places:
                        self._move(place)
                        return
                
                self._move()

            def _find_pair(self):
                for coord in self.swarm:
                    if self.world.entities[coord].sex != self.sex:
                        return True
                
                return False

            def _repr(self):
                coords = crusade(*self.coord)
                place = None

                for coord in coords:
                    if not coord in self.world:
                        continue
                    elif coord in self.world.free_places or find_parent_name(self.world.entities[coord]) != find_parent_name(self):
                        place = coord
                        break

                if place is None:
                    return
                
                new_animal = self.__class__(self.world, place)
                if place in self.world.free_places:
                    self.world.add_entity(new_animal)
                else:
                    self.world.repl_entity(new_animal)

            def _find_swarm(self):
                swarm = [self.coord]
                visited = set()
                queue = [self.coord]
                visited.add(self.coord)

                while queue:
                    current_coord = queue.pop(0)
                    neighbors = circle(*current_coord, r=2)

                    for neighbor in neighbors:
                        if neighbor not in visited:
                            visited.add(neighbor)
                            if (neighbor in self.world.entities and 
                                find_class_name(self.world.entities[neighbor]) == find_class_name(self)):
                                swarm.append(neighbor)
                                queue.append(neighbor)

                return swarm
            
            cls._adapt = _adapt
            cls._eat = _eat
            cls._hunt = _hunt
            cls._move = _move
            cls._move_to_swarm = _move_to_swarm
            cls._find_pair = _find_pair
            cls._repr = _repr
            cls._find_swarm = _find_swarm

            cls.diet = []
            cls.sex = 'G'
            cls.aggr = 0
            cls.hunger = 0
            cls.max_hunger = 150
            cls.max_swarm = 0
            cls.max_age = 480

        return cls

In [167]:
class Plant(Entity, metaclass=EcosystemMeta):
    def __init__(self, world, coord):
        super().__init__(world, coord)

class Demi(Plant):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "|\033[48;2;93;121;48;1m D \033[0m"
        self.active_time = ["Morning", "Afternoon"]

class Obscurite(Plant):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "|\033[48;5;58m O \033[0m"
        self.active_time = ["Night", "Afternoon"]

class Lumiere(Plant):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "|\033[48;2;78;154;64;1m L \033[0m"
        self.active_time = ["Morning", "Day"]

In [168]:
class Animal(Entity, metaclass=EcosystemMeta):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.sex = sex_gen()
        self.swarm = [coord]
    
class Pauvre(Animal):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "|\033[48;5;1m P \033[0m"
        self.active_time = ["Morning", "Day", "Afternoon"]
        self.diet = ["Lumiere"]
        self.max_swarm = 5

class Malheureux(Animal):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "|\033[48;2;128;0;32;1m M \033[0m"
        self.active_time = ["Morning", "Afternoon"]
        self.diet = ["Demi", "Obscurite", "Pauvre"]
        self.max_swarm = 7

# Тесты

In [169]:
game = GameLoop([(Demi, 4), (Obscurite, 4), (Lumiere, 4), (Malheureux, 2), (Pauvre, 2)], (20, 16))
game.loop(100, logging=True, log_filename="all")

Day 4 | 3:00 (Night)
.................................................................................
| O | O | L |   | D | D |   | D | D |   | D | D |   | D |   | D | D | D |   | D |   
| O | O |   | D | D | D |   | D |   | D |   | D | D | D |   |   | D | D | D | D |   
| O | O | D | D |   | L | D | D | D | D |   |   |   |   | D |   |   |   | D |   |   
| L | D | D |   | D | D |   | D |   | D | D | D | M | D | D |   |   |   |   | D |   
|   |   | D | D | D | D | O | D | D | D | D | D |   | M | D | D | D | D |   |   |   
|   | L | L | D | D | O | O | P | D | D |   | D | D |   | D | D | D | D | D |   |   
| L |   |   | D | O | O | O | O | D | D | D |   | D | D | D | D |   | D | D | M |   
|   |   | L |   |   | O |   |   | D |   | L | L | M |   | L | D | M |   | D | M |   
|   | D | D |   |   | O | O | O |   |   | M |   | L |   | L |   | D |   | M | D |   
|   | D | D | D | D | O | L | D | D |   |   |   |   | L |   |   | L |   | M |   |   
| D | D | D | D | D | D | D |   | D | D | L |  

In [170]:
import unittest

class TestEcosystemMeta(unittest.TestCase):

    def test_entities_registration(self):
        self.assertIn("Lumiere", PLANT_REGISTRY)
        self.assertIn("Obscurite", PLANT_REGISTRY)
        self.assertIn("Demi", PLANT_REGISTRY)
        self.assertIn("Pauvre", ANIMAL_REGISTRY)
        self.assertIn("Malheureux", ANIMAL_REGISTRY)

    def test_entities_have_methods(self):
        for plant_cls in [Lumiere, Obscurite, Demi]:
            plant = plant_cls(World((1, 1)), (0, 0))

            for method_name in ['_do_nothing', '_die', '_lifecycle', 'live', '_adapt', '_grow']:
                self.assertTrue(hasattr(plant, method_name))
                self.assertTrue(callable(getattr(plant, method_name)))
        
        for animal_cls in [Pauvre, Malheureux]:
            animal = animal_cls(World((1, 1)), (0, 0))

            for method_name in ['_do_nothing', '_die', '_lifecycle', 'live', '_adapt', '_eat',
                                '_hunt', '_move', '_move_to_swarm', '_find_pair', '_repr', '_find_swarm']:
                self.assertTrue(hasattr(animal, method_name))
                self.assertTrue(callable(getattr(animal, method_name)))

    def test_behavior_change_by_time(self):
        world = World((20, 20))
        times = {'Night': 0,
                 'Morning': 6,
                 'Day': 12,
                 'Afternoon': 18}

        for plant_cls in [Lumiere, Obscurite, Demi]:
            plant = plant_cls(world, (10, 10))

            for pos, time in times.items():
                world.cel_obj.ticks = time

                plant._adapt()

                if pos in plant.active_time:
                    self.assertEqual(plant._lifecycle, plant._grow)
                else:
                    self.assertEqual(plant._lifecycle, plant._do_nothing)

            plant.max_age = 0
            plant.age = float('inf')

            plant._adapt()
            self.assertEqual(plant._lifecycle, plant._die)
        
        for animal_cls in [Pauvre, Malheureux]:
            animal = animal_cls(world, (10, 10))

            animal.aggr = float('inf')
            animal.max_swarm = 0

            for pos, time in times.items():
                world.cel_obj.ticks = time

                world.repl_entity(animal)

                animal._adapt()

                if pos in animal.active_time:
                    self.assertNotEqual(animal._lifecycle, animal._do_nothing)
                else:
                    self.assertEqual(animal._lifecycle, animal._do_nothing)

            world.cel_obj.ticks = 18
            plant.max_age = 0
            plant.age = float('inf')

            plant._adapt()
            self.assertEqual(plant._lifecycle, plant._die)

unittest.main(argv=['first-arg-is-ignored'], exit=False)

...
----------------------------------------------------------------------
Ran 3 tests in 0.003s

OK
